# 🗓 Calendar Agent — Colab LoRA Training

Qwen2.5-0.5B 베이스를 합성 데이터(JSONL)로 LoRA 파인튜닝.

**전제**: Colab Runtime > Change runtime type > **T4 GPU** 선택.

**흐름**: clone repo → install deps → upload data → train → download adapter.

예상 소요: 1820건 × 3 epochs on T4 ≈ **10~20분**. 비용 0.

## 0. GPU 확인

아래 셀이 `Tesla T4` 또는 비슷한 16GB VRAM GPU를 보여야 함. "No GPU"가 뜨면 런타임 설정 다시 확인.

In [ ]:
!nvidia-smi

## 1. 레포 clone (Private 레포 — PAT 필요)

`soo-vibe/calendar-agent`는 Private이라 GitHub Personal Access Token(PAT)이 필요. 생성 방법:

1. https://github.com/settings/tokens/new
2. **Note**: `colab-clone` (아무 이름)
3. **Expiration**: 7 days 정도면 충분
4. **Scopes**: `repo` 만 체크
5. Generate → 한 번만 노출되니 복사해두고 아래 셀 실행 시 붙여넣기

PAT는 이 노트북 메모리에만 머무르고 디스크에는 안 남음.

In [ ]:
import os, getpass

if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/soo-vibe/calendar-agent.git
    token = None  # 메모리에서 비우기

%cd calendar-agent
!git log --oneline -1

## 2. 학습 의존성 설치

Colab은 PyTorch+CUDA가 미리 설치돼 있음. `pip install -e .[train]`이 transformers/peft/trl/accelerate/bitsandbytes/datasets 등을 추가 설치. 2~3분 소요.

In [ ]:
!pip install -q -e .[train]
print('install done')
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## 3. 데이터 업로드

`data/raw/`와 `data/processed/`는 gitignore되어 있어서 Colab에 별도 업로드 필요.
필요 파일:
- `data/processed/v1_train.jsonl` (1639건)
- `data/processed/v1_val.jsonl` (181건)
- `data/eval/golden.jsonl` (5건)

**두 가지 방법 중 하나 선택**:
- **A. Drive 마운트** (반복 학습 시 추천) — 사전에 `MyDrive/calendar-agent-data/`에 3개 파일 업로드해두기
- **B. 노트북 직접 업로드** — 일회성

### 옵션 A — Google Drive 마운트

In [ ]:
# 사용 시 주석 해제
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/processed data/eval
# !cp /content/drive/MyDrive/calendar-agent-data/v1_train.jsonl data/processed/
# !cp /content/drive/MyDrive/calendar-agent-data/v1_val.jsonl data/processed/
# !cp /content/drive/MyDrive/calendar-agent-data/golden.jsonl data/eval/

### 옵션 B — 노트북 직접 업로드

셀 실행 후 "파일 선택" 다이얼로그가 뜨면 3개 파일 한꺼번에 선택.

In [ ]:
# 사용 시 주석 해제
# from google.colab import files
# import shutil, os
# os.makedirs('data/processed', exist_ok=True)
# os.makedirs('data/eval', exist_ok=True)
# uploaded = files.upload()
# for fn in uploaded:
#     if 'train' in fn:
#         shutil.move(fn, 'data/processed/v1_train.jsonl')
#     elif 'val' in fn:
#         shutil.move(fn, 'data/processed/v1_val.jsonl')
#     elif 'golden' in fn:
#         shutil.move(fn, 'data/eval/golden.jsonl')
# print('uploaded:', list(uploaded.keys()))

## 4. 데이터 sanity check

In [ ]:
import orjson
from pathlib import Path
for p in [Path('data/processed/v1_train.jsonl'), Path('data/processed/v1_val.jsonl'), Path('data/eval/golden.jsonl')]:
    assert p.exists(), f'MISSING: {p}'
    n = sum(1 for line in open(p, 'rb') if line.strip())
    print(f'{p}: {n} records')
print('OK')

## 5. WandB 끄기 + bf16 설정 확인

T4는 bf16 미지원 → fp16으로 전환.  Ampere(A100/A6000) 이상이라면 bf16 그대로 OK.

In [ ]:
import torch
supports_bf16 = torch.cuda.is_bf16_supported()
print(f'GPU supports bf16: {supports_bf16}')

# wandb 끄기 + T4면 fp16로 전환
!sed -i 's/report_to: wandb/report_to: none/' configs/train.yaml
if not supports_bf16:
    !sed -i 's/bf16: true/bf16: false/' configs/train.yaml
    !sed -i 's/fp16: false/fp16: true/' configs/train.yaml
    !sed -i "s|bnb_4bit_compute_dtype: bfloat16|bnb_4bit_compute_dtype: float16|" configs/train.yaml
    print('switched to fp16')

!grep -E 'report_to|bf16|fp16|bnb_4bit_compute' configs/train.yaml

## 6. 학습 실행

1820건 × 3 epochs.  T4 기준 ~10-20분.
로그가 길면 `print` 위주만 보고 spinner는 무시.

In [ ]:
!python scripts/train_lora.py --config configs/train.yaml

## 7. 결과 패키징 + 다운로드

LoRA adapter는 보통 20~40MB. 로컬로 다운받아 merge → quantize.

In [ ]:
!ls -la models/lora/v1/
!zip -r lora_v1.zip models/lora/v1 configs/train.yaml configs/lora.yaml configs/model_qwen.yaml
from google.colab import files
files.download('lora_v1.zip')

## (선택) Drive로도 백업

다음 세션에서 이어쓰거나 분실 방지용.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
# !cp lora_v1.zip /content/drive/MyDrive/